In [ ]:
from feature_extraction_functions import *
import pandas as pd 

In [ ]:
ROOT_DIR = r"Datos/Originales/Datos/03_Validacion"

metadata = build_metadata_table(ROOT_DIR)
metadata.head()

,file_id,file_path,is_anomaly,fault_type,severity
0,0,data\horizontal-misalignment\0.5mm\12.288.csv,1,horizontal-misalignment,0.5mm
1,1,data\horizontal-misalignment\0.5mm\13.5168.csv,1,horizontal-misalignment,0.5mm
2,2,data\horizontal-misalignment\0.5mm\14.5408.csv,1,horizontal-misalignment,0.5mm
3,3,data\horizontal-misalignment\0.5mm\15.36.csv,1,horizontal-misalignment,0.5mm
4,4,data\horizontal-misalignment\0.5mm\16.384.csv,1,horizontal-misalignment,0.5mm


In [4]:
metadata.shape

(1951, 5)

In [3]:
metadata['is_anomaly'].value_counts()
(metadata['is_anomaly'].value_counts(normalize=True) * 100).round(2)

is_anomaly
1    97.49
0     2.51
Name: proportion, dtype: float64

In [8]:
metadata['fault_type'].value_counts()
(metadata['fault_type'].value_counts(normalize=True) * 100).round(2)

fault_type
imbalance                  17.07
vertical-misalignment      15.43
horizontal-misalignment    10.10
overhang_cage_fault         9.64
overhang_outer_race         9.64
underhang_cage_fault        9.64
underhang_ball_fault        9.53
underhang_outer_race        9.43
overhang_ball_fault         7.02
normal                      2.51
Name: proportion, dtype: float64

In [10]:
metadata.groupby('fault_type')['severity'].value_counts()
metadata.groupby('fault_type')['severity'].value_counts(normalize=True).round(3)

fault_type               severity
horizontal-misalignment  0.5mm       0.254
                         1.0mm       0.249
                         1.5mm       0.249
                         2.0mm       0.249
imbalance                6g          0.147
                         20g         0.147
                         15g         0.144
                         10g         0.144
                         30g         0.141
                         25g         0.141
                         35g         0.135
normal                   normal      1.000
overhang_ball_fault      0g          0.358
                         6g          0.314
                         20g         0.182
                         35g         0.146
overhang_cage_fault      0g          0.261
                         20g         0.261
                         6g          0.261
                         35g         0.218
overhang_outer_race      6g          0.261
                         0g          0.261
                    

In [4]:
WINDOW_SIZE = 4096
HOP_SIZE = 2048
FS = 51200

BANDS = [
    (0, 1000),
    (1000, 5000),
    (5000, 12000)
]

In [11]:
time_dataset_df = build_feature_dataset(
    metadata=metadata,
    window_size=WINDOW_SIZE,
    hop_size=HOP_SIZE,
    feature_func=time_features,
    feature_names_func=get_time_feature_names
)

In [12]:
time_dataset_df.head()

,file_id,is_anomaly,fault_type,severity,acc_under_axial_mean,acc_under_axial_std,acc_under_axial_rms,acc_under_axial_peak_to_peak,acc_under_axial_max_abs,acc_under_axial_kurtosis,...,acc_over_tangencial_rms,acc_over_tangencial_peak_to_peak,acc_over_tangencial_max_abs,acc_over_tangencial_kurtosis,microphone_mean,microphone_std,microphone_rms,microphone_peak_to_peak,microphone_max_abs,microphone_kurtosis
0,0,1,horizontal-misalignment,0.5mm,-0.391321,1.256883,1.316392,5.6725,4.2005,2.138032,...,0.209716,0.804760,0.59175,2.788997,0.018394,0.140553,0.141751,1.05629,0.76796,4.907437
1,0,1,horizontal-misalignment,0.5mm,-0.138545,1.155537,1.163813,5.4420,3.9081,2.410662,...,0.542621,1.267380,1.09710,1.829811,0.011347,0.142445,0.142896,0.97422,0.67489,4.541004
2,0,1,horizontal-misalignment,0.5mm,0.348664,0.866709,0.934212,4.8077,3.2369,2.915426,...,0.542464,1.574960,1.09710,1.902854,0.014962,0.146347,0.147110,0.97422,0.67489,4.099488
3,0,1,horizontal-misalignment,0.5mm,0.375923,0.815721,0.898175,4.0834,2.5126,2.644776,...,0.334343,1.279940,0.69770,2.246651,0.025450,0.145196,0.147409,0.92846,0.63271,3.171471
4,0,1,horizontal-misalignment,0.5mm,-0.092894,1.067403,1.071438,4.6776,3.2369,2.157993,...,0.405232,0.703584,0.73516,2.254136,0.022664,0.135233,0.137119,0.89512,0.63271,3.229024


In [13]:
time_dataset_df.shape

(236071, 46)

In [ ]:
time_dataset_df.to_csv("Datos/Transformados/SAD/time_dataset_df.csv", index=False)

In [14]:
freq_dataset_df = build_feature_dataset(
    metadata=metadata,
    window_size=WINDOW_SIZE,
    hop_size=HOP_SIZE,
    feature_func=spectral_features,
    feature_names_func=get_spectral_feature_names,
    fs=FS,
    bands=BANDS
)

In [ ]:
freq_dataset_df.to_csv("Datos/Transformados/SAD/freq_dataset_df.csv", index=False)

In [8]:
stft_dataset_df = build_feature_dataset(
    metadata=metadata,
    window_size=WINDOW_SIZE,
    hop_size=HOP_SIZE,
    feature_func=stft_features,
    feature_names_func=get_stft_feature_names,
    fs=FS,
    bands=BANDS,
    nperseg=512,
    noverlap=256
)

In [ ]:
stft_dataset_df.to_csv("Datos/Transformados/SAD/stft_dataset_df.csv", index=False)